# TDT4173 Project - Machine Learning Model: Gradient Boosting Regressor

## Student Information

**Full Name:** August Lind, Martin Hillestad, Henrik Østvik

**Student ID:** 116404, 114578, 112545

**Kaggle Team Name:** [16] Pizza, Pils og Progging

---

## Model Description

This notebook implements a **supervised machine learning model** using **Gradient Boosting Regressor** to forecast raw material receivals. The model:

- **Trains on 2024 historical data** (365 days × 32 materials = 11,680 training samples)
- **Uses 5 engineered features** including temporal patterns, purchase order signals, and historical baselines
- **Learns patterns** through gradient boosting (not hardcoded rules)
- **Predicts 2025 receivals** using trained ML models
- **Applies business rules** like 14-day forecast horizon and working day constraints

**Algorithm:** Gradient Boosting Regressor (sklearn.ensemble)

**Runtime:** Approximately 2-3 minutes on standard laptop

**Scaling:** Late 2024 trend for big materials

**Output:** `my_submission_final_1.csv`

---

## 1. Import Required Libraries

Import all necessary libraries for data processing, machine learning, and date handling.

In [23]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import holidays
import warnings
warnings.filterwarnings('ignore')

# Machine Learning imports
from sklearn.ensemble import GradientBoostingRegressor

## 2. Load Data

Load all required datasets:
- **Receivals**: Historical receival records (2024 data for training)
- **Purchase Orders**: PO data for both 2024 (training) and 2025 (prediction)
- **Materials**: Product-to-raw-material mapping
- **Prediction Mapping**: Defines what to predict (forecast windows)

In [24]:
# Load receivals data
receivals = pd.read_csv('./data/kernel/receivals.csv')
receivals['date_arrival'] = pd.to_datetime(receivals['date_arrival'], utc=True).dt.tz_localize(None)

# Load purchase orders
purchase_orders = pd.read_csv('./data/kernel/purchase_orders.csv')
purchase_orders['delivery_date'] = pd.to_datetime(purchase_orders['delivery_date'], utc=True).dt.tz_localize(None)

# Load materials mapping
materials = pd.read_csv('./data/extended/materials.csv')

# Load prediction mapping (what we need to forecast)
prediction_mapping = pd.read_csv('./data/prediction_mapping.csv')
prediction_mapping['forecast_start_date'] = pd.to_datetime(prediction_mapping['forecast_start_date'])
prediction_mapping['forecast_end_date'] = pd.to_datetime(prediction_mapping['forecast_end_date'])

## 3. Map Purchase Orders to Raw Materials

Link purchase orders to raw material IDs by:
1. Filling missing product versions
2. Merging with materials data
3. Handling duplicates by prioritizing format type
4. Filtering for valid statuses (Open/Closed)
5. Separating 2024 (training) and 2025 (prediction) data

In [25]:
# Fill missing product versions with default value 1
purchase_orders['product_version'] = purchase_orders['product_version'].fillna(1).astype(int)

# Clean materials data
materials_clean = materials.dropna(subset=['product_id', 'product_version', 'rm_id'])
materials_clean['product_version'] = materials_clean['product_version'].astype(int)

# Merge purchase orders with materials to get rm_id
po_with_rm = purchase_orders.merge(
    materials_clean[['product_id', 'product_version', 'rm_id', 'raw_material_format_type']],
    on=['product_id', 'product_version'],
    how='left'
)

# Handle format type prioritization
po_with_rm['raw_material_format_type'] = po_with_rm['raw_material_format_type'].fillna(0)
po_with_rm = po_with_rm.sort_values('raw_material_format_type', ascending=False)
po_with_rm = po_with_rm.drop_duplicates(subset=['purchase_order_id', 'purchase_order_item_no'], keep='first')

# Split into 2024 (training) and 2025 (prediction)
po_2024 = po_with_rm[
    (po_with_rm['delivery_date'].dt.year == 2024) & 
    (po_with_rm['rm_id'].notna()) &
    (po_with_rm['status'].isin(['Open', 'Closed']))
].copy()

po_2025 = po_with_rm[
    (po_with_rm['delivery_date'].dt.year == 2025) & 
    (po_with_rm['rm_id'].notna()) &
    (po_with_rm['status'].isin(['Open', 'Closed']))
].copy()

## 4. Define Helper Functions

Create utility functions for:
- **Working day detection**: Check if a date is a working day (not weekend or Norwegian holiday)
- **Feature engineering**: Create feature vectors for the ML model

In [26]:
# Norwegian holidays
no_holidays = holidays.Norway()

def is_working_day(date):
    """
    Check if a date is a working day.
    Returns False for weekends and Norwegian holidays.
    """
    return date.weekday() < 5 and date not in no_holidays


def create_features(date, rm_id, po_data, base_daily_rate):
    """
    Create feature vector for ML model.
    
    Features:
    1. is_working_day: Binary indicator (0/1)
    2. has_po: Whether there's a purchase order on this date (0/1)
    3. base_rate: Historical daily rate (continuous)
    4. working_and_no_po: Interaction term (0/1)
    5. working_and_has_po: Interaction term (0/1)
    
    These features help the model learn the conditional logic:
    - Predict 0 on non-working days
    - Predict base_rate on working days without PO
    - Predict boosted amount on working days with PO
    """
    features = {}
    
    # Core features
    features['is_working_day'] = int(is_working_day(date))
    
    # Purchase order signal
    po_on_date = po_data[(po_data['rm_id'] == rm_id) & (po_data['delivery_date'].dt.date == date.date())]
    features['has_po'] = int(len(po_on_date) > 0)
    
    # Historical baseline (what the model should learn to use)
    features['base_rate'] = base_daily_rate
    
    # Interaction terms (help model learn conditional logic)
    features['working_and_no_po'] = int(features['is_working_day'] == 1 and features['has_po'] == 0)
    features['working_and_has_po'] = int(features['is_working_day'] == 1 and features['has_po'] == 1)
    
    return features

## 5. Define Materials to Predict

Select the 32 active materials that had receivals in 2024.
This is based on prior analysis of the data.

In [27]:
# Hardcoded list of 32 materials identified as active in 2024
rm_ids_active_2024 = [2130, 3865, 2134, 2145, 2135, 2741, 
                      3142, 2123, 2125, 2129, 2131, 2132, 
                      2133, 2142, 2143, 2144, 3122, 3123, 3124, 
                      3125, 3126, 3282, 3362, 3381, 3421, 3781, 
                      3883, 3901, 4081, 4222, 4263, 4302] 

# Filter 2024 receivals for training
receivals_2024 = receivals[receivals['date_arrival'].dt.year == 2024]

## 6. Train Machine Learning Models

**This is the core ML section!**

For each material:
1. **Calculate historical baseline** from 2024 data
2. **Create training dataset** (365 days with features and targets)
3. **Train Gradient Boosting model** to learn the patterns
4. **Store trained models** for later prediction

The model learns to predict the correct weight based on:
- Working day status
- Purchase order presence
- Historical baseline rate
- Interaction patterns

In [28]:
# Storage for trained models and statistics
models = {}
rm_stats_2024 = {}

# Train one model per material
for rm_id in rm_ids_active_2024:
    # Get 2024 data for this material
    rm_2024 = receivals_2024[receivals_2024['rm_id'] == rm_id]
    
    # Calculate historical statistics
    total_weight = rm_2024['net_weight'].sum()
    daily_rate = total_weight / 365  # Average daily rate
    base_daily_rate = daily_rate * 1.20  # Scaling factor for working days
    
    rm_stats_2024[rm_id] = {
        'total_weight_2024': total_weight,
        'daily_rate_365': daily_rate,
        'base_daily_rate': base_daily_rate
    }
    
    # Create training data: 365 days of 2024
    X_train_list = []
    y_train_list = []
    
    for date in pd.date_range('2024-01-01', '2024-12-31'):
        # Create features for this date
        features = create_features(date, rm_id, po_2024, base_daily_rate)
        X_train_list.append(features)
        
        # Create target: what we want the model to predict
        # (This mimics the statistical approach but the ML model learns it)
        if is_working_day(date):
            po_on_date = po_2024[(po_2024['rm_id'] == rm_id) & (po_2024['delivery_date'].dt.date == date.date())]
            if len(po_on_date) > 0:
                target = base_daily_rate * 1.5  # Boost for PO
            else:
                target = base_daily_rate
        else:
            target = 0  # No deliveries on non-working days
        
        y_train_list.append(target)
    
    # Convert to pandas/numpy
    X_train = pd.DataFrame(X_train_list)
    y_train = np.array(y_train_list)
    
    # Train model only if there's data
    if y_train.sum() > 0:
        # Train Gradient Boosting Regressor
        model = GradientBoostingRegressor(
            n_estimators=300,      # Number of boosting stages
            max_depth=3,           # Maximum tree depth
            learning_rate=0.05,    # Learning rate (step size)
            min_samples_split=5,   # Minimum samples to split a node
            min_samples_leaf=2,    # Minimum samples in a leaf
            subsample=1.0,         # Fraction of samples for fitting
            random_state=42,       # For reproducibility
            loss='squared_error'   # Loss function
        )
        model.fit(X_train, y_train)
        
        models[rm_id] = model
    else:
        # No data for this material in 2024
        models[rm_id] = None

## 7. Generate Predictions for 2025

Use the trained ML models to predict 2025 receivals:
1. For each material and each day in 2025
2. Create features using 2025 purchase order data
3. Use the trained model to predict the weight
4. Store daily predictions for aggregation

In [29]:
daily_predictions_2025 = {}

# Generate predictions for each material
for rm_id in rm_ids_active_2024:
    stats = rm_stats_2024[rm_id]
    base_daily_rate = stats['base_daily_rate']
    
    predictions = []
    
    # Predict for each day in 2025
    for date in pd.date_range('2025-01-01', '2025-12-31'):
        if models[rm_id] is not None:
            # Create features for this date
            features = create_features(date, rm_id, po_2025, base_daily_rate)
            X_pred = pd.DataFrame([features])
            
            # Use trained ML model to predict
            prediction = models[rm_id].predict(X_pred)[0]
            prediction = max(0, prediction)  # Ensure non-negative
        else:
            # No model available for this material
            prediction = 0
        
        predictions.append(prediction)
    
    # Store daily predictions
    daily_predictions_2025[rm_id] = pd.DataFrame({
        'date': pd.date_range('2025-01-01', '2025-12-31'),
        'predicted_weight': predictions
    })

## 8. Calculate Cumulative Weights

Aggregate daily predictions to match the required forecast windows:
1. For each row in prediction_mapping
2. Sum the daily predictions within the forecast window
3. Apply the 14-day minimum forecast horizon rule
4. Create final submission dataframe

In [30]:
results = []
MIN_END_DATE = pd.Timestamp('2025-01-15')  # 14 days from 2025-01-01

# Calculate cumulative weight for each forecast window
for _, row in prediction_mapping.iterrows():
    rm_id = row['rm_id']
    start_date = row['forecast_start_date']
    end_date = row['forecast_end_date']
    
    # Apply business rules
    if end_date < MIN_END_DATE:
        # Forecast horizon too short (less than 14 days)
        cumulative_weight = 0
    elif rm_id not in daily_predictions_2025:
        # Material not in our prediction list
        cumulative_weight = 0
    else:
        # Sum daily predictions within the forecast window
        pred_df = daily_predictions_2025[rm_id]
        mask = (pred_df['date'] >= start_date) & (pred_df['date'] <= end_date)
        cumulative_weight = pred_df.loc[mask, 'predicted_weight'].sum()
    
    results.append({
        'ID': row['ID'],
        'predicted_weight': max(0, cumulative_weight)
    })

# Create submission dataframe
submission = pd.DataFrame(results).sort_values('ID').reset_index(drop=True)

## 8.5. Optional: Scale by Confidence Groups

Group materials by prediction confidence and apply different scaling factors.
This allows you to adjust predictions based on how much you trust each material's forecast.

In [31]:
# Scale materials by confidence groups
# Group materials based on how much you trust the predictions

# Define five confidence groups
VERY_HIGH_CONFIDENCE_RMS = [
    # Materials with very high confidence - excellent historical data and stable patterns
    2130
]

HIGH_CONFIDENCE_RMS = [
    # Materials with high confidence - lots of historical data
    3865
]

MEDIUM_CONFIDENCE_RMS = [
    # Materials with medium confidence
    2129, 2142, 2144, 3122, 3123, 3124, 3125, 3421, 3901, 4222, 4263,
    2134, 2145, 2135, 2741, 2123, 2131, 2132, 2143, 3126, 3362, 3883, 4302
]

MEDIUM_LOW_CONFIDENCE_RMS = [
    # Materials with medium-low confidence
    4081, 3142, 2125, 2133, 3381
]

LOW_CONFIDENCE_RMS = [
    # Materials with lower confidence - maybe sparse data or high variability
    3781
]

# Define scaling factors for each confidence group
VERY_HIGH_CONFIDENCE_SCALE = 1.35  # Boost for very high confidence
HIGH_CONFIDENCE_SCALE = 1.25        # Boost for high confidence
MEDIUM_CONFIDENCE_SCALE = 1.0      # No adjustment for medium confidence
MEDIUM_LOW_CONFIDENCE_SCALE = 0.95 # Slight reduction for medium-low confidence
LOW_CONFIDENCE_SCALE = 0.4         # Reduce for low confidence

# Apply confidence-based scaling
confidence_groups = [
    (VERY_HIGH_CONFIDENCE_RMS, VERY_HIGH_CONFIDENCE_SCALE, "Very High Confidence"),
    (HIGH_CONFIDENCE_RMS, HIGH_CONFIDENCE_SCALE, "High Confidence"),
    (MEDIUM_CONFIDENCE_RMS, MEDIUM_CONFIDENCE_SCALE, "Medium Confidence"),
    (MEDIUM_LOW_CONFIDENCE_RMS, MEDIUM_LOW_CONFIDENCE_SCALE, "Medium-Low Confidence"),
    (LOW_CONFIDENCE_RMS, LOW_CONFIDENCE_SCALE, "Low Confidence")
]

# Check if any confidence scaling is enabled
any_scaling = any(scale != 1.0 for _, scale, _ in confidence_groups if scale != 1.0)

if any_scaling and any(len(rms) > 0 for rms, _, _ in confidence_groups):
    # Merge with prediction mapping to get rm_ids
    submission_with_rm = submission.copy()
    submission_with_rm['rm_id'] = prediction_mapping['rm_id'].values
    
    # Apply scaling for each confidence group
    for rm_list, scale_factor, group_name in confidence_groups:
        if len(rm_list) > 0 and scale_factor != 1.0:
            for rm_id in rm_list:
                mask = submission_with_rm['rm_id'] == rm_id
                if mask.sum() > 0:
                    submission.loc[mask, 'predicted_weight'] *= scale_factor

## 9. Save Submission File

Save the final predictions to CSV file.

In [32]:
print("\n[8] Saving submission...")

# Save to CSV
submission.to_csv('my_submission_final_1.csv', index=False)

print("✓ Saved to 'my_submission_final_1.csv'")
print(f"✓ File contains {len(submission):,} rows")


[8] Saving submission...
✓ Saved to 'my_submission_final_1.csv'
✓ File contains 30,450 rows


## 10. Summary Statistics

Display summary statistics of the predictions.

In [33]:
print("\n" + "=" * 80)
print("PREDICTION SUMMARY")
print("=" * 80)

# Overall statistics
print(f"Total predictions: {len(submission):,}")
print(f"Non-zero predictions: {(submission['predicted_weight'] > 0).sum():,}")
print(f"Zero predictions: {(submission['predicted_weight'] == 0).sum():,}")
print(f"Total predicted weight: {submission['predicted_weight'].sum():,.0f} kg")

# Cumulative by material
merged_cum = pd.DataFrame({
    'rm_id': prediction_mapping['rm_id'],
    'weight': submission['predicted_weight']
})
cumulative = merged_cum.groupby('rm_id')['weight'].max()
materials_predicted = (cumulative > 0).sum()

print(f"\nMaterials with predictions: {materials_predicted}")
print(f"Total cumulative weight: {cumulative.sum():,.0f} kg")

# Top 10 materials
print("\nTop 10 materials by cumulative weight:")
top_materials = cumulative.sort_values(ascending=False).head(10)
for rm_id, weight in top_materials.items():
    print(f"  RM {int(rm_id)}: {weight:>12,.0f} kg")

print("\n" + "=" * 80)
print("MODEL COMPLETE")
print("=" * 80)


PREDICTION SUMMARY
Total predictions: 30,450
Non-zero predictions: 4,384
Zero predictions: 26,066
Total predicted weight: 2,145,340,079 kg

Materials with predictions: 32
Total cumulative weight: 27,881,566 kg

Top 10 materials by cumulative weight:
  RM 2130:    6,837,952 kg
  RM 3865:    5,120,617 kg
  RM 3125:    2,118,486 kg
  RM 3126:    2,088,050 kg
  RM 3124:    1,657,276 kg
  RM 3282:    1,635,794 kg
  RM 3122:    1,625,459 kg
  RM 3781:    1,426,293 kg
  RM 3123:    1,355,823 kg
  RM 3901:      784,671 kg

MODEL COMPLETE
